In [52]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.svm import SVC

from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score
)

# ==================================================
# LOAD DATA
# ==================================================

df = pd.read_csv("Breast_Cancer_METABRIC.csv")

# ==================================================
# CREATE TARGET
# ==================================================

def create_target(row):

    survival = row["Overall Survival (Months)"]

    alive = str(
        row["Patient's Vital Status"]
    ).lower()

    relapse = str(
        row["Relapse Free Status"]
    ).lower()

    if (
        survival >= 60
        and "living" in alive
        and "not" in relapse
    ):
        return 1

    return 0

df["Effective_Treatment"] = df.apply(
    create_target,
    axis=1
)

# ==================================================
# FEATURES
# ==================================================

features = [

    "Age at Diagnosis",

    "Tumor Size",

    "Tumor Stage",

    "Neoplasm Histologic Grade",

    "Lymph nodes examined positive",

    "ER Status",

    "PR Status",

    "HER2 Status",

    "Inferred Menopausal State",

    "Chemotherapy",

    "Hormone Therapy",

    "Radio Therapy"
]

X = df[features]

y = df["Effective_Treatment"]

# ==================================================
# TRAIN TEST SPLIT
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==================================================
# NUMERIC FEATURES
# ==================================================

num_features = [

    "Age at Diagnosis",

    "Tumor Size",

    "Tumor Stage",

    "Neoplasm Histologic Grade",

    "Lymph nodes examined positive"
]

# ==================================================
# CATEGORICAL FEATURES
# ==================================================

cat_features = [

    "ER Status",

    "PR Status",

    "HER2 Status",

    "Inferred Menopausal State",

    "Chemotherapy",

    "Hormone Therapy",

    "Radio Therapy"
]

# ==================================================
# PREPROCESSOR
# ==================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),

                (
                    "scaler",
                    StandardScaler()
                )

            ]),

            num_features
        ),

        (
            "cat",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )

            ]),

            cat_features
        )
    ]
)

# ==================================================
# SVM
# ==================================================

base_svm = SVC(
    kernel="rbf",
    class_weight="balanced"
)

svm = CalibratedClassifierCV(
    estimator=base_svm,
    cv=5
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", svm)
])

# ==================================================
# GRID SEARCH
# ==================================================

param_grid = {

    "classifier__estimator__C":
        [1,10,100],

    "classifier__estimator__gamma":
        [0.001,0.01,0.1]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

grid.fit(
    X_train,
    y_train
)

best_model = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

# ==================================================
# MODEL PERFORMANCE
# ==================================================

y_prob = best_model.predict_proba(X_test)[:,1]

y_pred = (
    y_prob >= 0.3
).astype(int)

print("\nAccuracy:",
      accuracy_score(
          y_test,
          y_pred
      ))

print(
    "\nROC AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

# ==================================================
# TREATMENT RECOMMENDER
# ==================================================

def recommend_treatment(
    patient,
    model
):

    treatment_combinations = list(
        itertools.product(
            ["Yes","No"],
            ["Yes","No"],
            ["Yes","No"]
        )
    )

    results = []

    for chemo, hormone, radio in treatment_combinations:

        patient_copy = patient.copy()

        patient_copy["Chemotherapy"] = chemo
        patient_copy["Hormone Therapy"] = hormone
        patient_copy["Radio Therapy"] = radio

        patient_df = pd.DataFrame(
            [patient_copy]
        )

        probability = model.predict_proba(
            patient_df
        )[0,1]

        results.append({

            "Chemotherapy":
                chemo,

            "Hormone Therapy":
                hormone,

            "Radio Therapy":
                radio,

            "Predicted Success Probability":
                probability
        })

    results = pd.DataFrame(
        results
    )

    results = results.sort_values(
        "Predicted Success Probability",
        ascending=False
    )

    return results

# ==================================================
# EXAMPLE PATIENT
# ==================================================

new_patient = {

    "Age at Diagnosis": 55,

    "Tumor Size": 25,

    "Tumor Stage": 2,

    "Neoplasm Histologic Grade": 2,

    "Lymph nodes examined positive": 1,

    "ER Status": "Positive",

    "PR Status": "Positive",

    "HER2 Status": "Negative",

    "Inferred Menopausal State": "Post"
}

# ==================================================
# RECOMMEND
# ==================================================

recommendations = recommend_treatment(
    new_patient,
    best_model
)

# ==================================================
# EVALUATE RECOMMENDATION ENGINE
# ==================================================

import numpy as np

recommended_treatments = []
recommended_probs = []

for idx, row in X_test.iterrows():

    # Remove actual treatment information
    patient = row.copy()

    patient = patient.drop([
        "Chemotherapy",
        "Hormone Therapy",
        "Radio Therapy"
    ])

    recs = recommend_treatment(
        patient,
        best_model
    )

    best_rec = recs.iloc[0]

    recommended_treatments.append(
        (
            best_rec["Chemotherapy"],
            best_rec["Hormone Therapy"],
            best_rec["Radio Therapy"]
        )
    )

    recommended_probs.append(
        best_rec["Predicted Success Probability"]
    )

# ==================================================
# RESULTS DATAFRAME
# ==================================================

results = X_test.copy().reset_index(drop=True)

results["Actual_Outcome"] = (
    y_test.reset_index(drop=True)
)

# Recommended treatments

results["Recommended_Chemo"] = [
    x[0] for x in recommended_treatments
]

results["Recommended_Hormone"] = [
    x[1] for x in recommended_treatments
]

results["Recommended_Radio"] = [
    x[2] for x in recommended_treatments
]

results["Recommended_Probability"] = (
    recommended_probs
)

# Actual treatment

results["Actual_Treatment"] = (

    X_test["Chemotherapy"]
    .reset_index(drop=True)

    + "_"

    + X_test["Hormone Therapy"]
    .reset_index(drop=True)

    + "_"

    + X_test["Radio Therapy"]
    .reset_index(drop=True)
)

# Recommended treatment

results["Recommended_Treatment"] = (

    results["Recommended_Chemo"]

    + "_"

    + results["Recommended_Hormone"]

    + "_"

    + results["Recommended_Radio"]
)

# ==================================================
# AGREEMENT
# ==================================================

agreement = (

    results["Actual_Treatment"]

    ==

    results["Recommended_Treatment"]

).mean()

print("\n")
print("=" * 60)
print("RECOMMENDATION AGREEMENT")
print("=" * 60)

print(
    f"Agreement: {agreement:.2%}"
)

evaluation = (
    results.groupby("Recommended_Treatment")
    ["Actual_Outcome"]
    .mean()
    .sort_values(ascending=False)
)

print(evaluation)

# ==================================================
# RECOMMENDATION SYSTEM SUMMARY
# ==================================================

print("\n")
print("=" * 60)
print("RECOMMENDATION SYSTEM SUMMARY")
print("=" * 60)

print(
    "Average Recommended Probability:",
    round(
        np.mean(recommended_probs),
        4
    )
)

print(
    "Median Recommended Probability:",
    round(
        np.median(recommended_probs),
        4
    )
)

print(
    "Maximum Recommended Probability:",
    round(
        np.max(recommended_probs),
        4
    )
)

print(
    "Minimum Recommended Probability:",
    round(
        np.min(recommended_probs),
        4
    )
)

# ==================================================
# MOST COMMON RECOMMENDATIONS
# ==================================================

print("\n")
print("=" * 60)
print("MOST COMMON RECOMMENDATIONS")
print("=" * 60)

print(
    results[
        "Recommended_Treatment"
    ].value_counts()
)

# ==================================================
# TREATMENT SUMMARY TABLE
# ==================================================

summary = (

    results.groupby(
        "Recommended_Treatment"
    )["Recommended_Probability"]

    .agg([
        "count",
        "mean",
        "min",
        "max"
    ])

    .sort_values(
        "mean",
        ascending=False
    )
)

print("\n")
print("=" * 60)
print("TREATMENT SUMMARY")
print("=" * 60)

print(summary)

# ==================================================
# TOP 20 PATIENTS
# ==================================================

print("\n")
print("=" * 60)
print("TOP RECOMMENDATIONS")
print("=" * 60)

print(

    results[
        [
            "Recommended_Chemo",
            "Recommended_Hormone",
            "Recommended_Radio",
            "Recommended_Probability"
        ]
    ]

    .head(20)

)

# ==================================================
# SAVE FILES
# ==================================================

results.to_csv(
    "svm_treatment_recommendations.csv",
    index=False
)

summary.to_csv(
    "treatment_summary.csv"
)

print("\nFiles saved:")
print(" - svm_treatment_recommendations.csv")
print(" - treatment_summary.csv")

# Save trained model

joblib.dump(
    best_model,
    "svm_breast_cancer_recommender.pkl"
)

print(
    "\nModel saved as svm_breast_cancer_recommender.pkl"
)


Best Parameters:
{'classifier__estimator__C': 100, 'classifier__estimator__gamma': 0.01}

Accuracy: 0.7051792828685259

ROC AUC: 0.7578551912568305
              precision    recall  f1-score   support

           0       0.86      0.71      0.78       366
           1       0.47      0.68      0.56       136

    accuracy                           0.71       502
   macro avg       0.66      0.70      0.67       502
weighted avg       0.75      0.71      0.72       502



RECOMMENDATION AGREEMENT
Agreement: 5.78%
Recommended_Treatment
No_Yes_No      0.444444
No_Yes_Yes     0.341463
Yes_Yes_No     0.321951
Yes_Yes_Yes    0.288462
Yes_No_Yes     0.103896
Yes_No_No      0.054054
No_No_Yes      0.000000
Name: Actual_Outcome, dtype: float64


RECOMMENDATION SYSTEM SUMMARY
Average Recommended Probability: 0.4223
Median Recommended Probability: 0.4251
Maximum Recommended Probability: 0.7709
Minimum Recommended Probability: 0.0187


MOST COMMON RECOMMENDATIONS
Recommended_Treatment
Yes_Yes_No